In [2]:
import torch
import numpy as np
from model.model import load_model
from dataset.dataset import load_data, create_dataloaders
from functions.optimizer import load_optimizer
from functions.loss import load_loss_fun
from functions.functions import eval_model, save_checkpoint, load_checkpoint
from functions.xai import explain_dataset, evaluate_explainations
from utils.utils import enable_reproducibility
from typing import Callable, Tuple
import torch.nn as nn
from torch.utils.data.dataloader import DataLoader
from torch.optim import Optimizer
from tqdm import tqdm
from functions.loss import load_loss_fun
import os
from typing import Optional
import matplotlib.pyplot as plt
import numpy as np
import torch.nn.functional as F
from experiments.utils import compute_correlations, compute_auc_roc,log_corr_results, log_auc_results
from functions.xil import compute_simplicity
from functions.functions import train_model
from functions.xai import explain_dataset,visualize_k_expl, evaluate_explainations

In [3]:
use_cuda = torch.cuda.is_available()
device = 'cuda' if use_cuda else 'cpu'
enable_reproducibility(123)

model = load_model("ResNet", model_name="resnet50", n_classes=2, device=device)
#RESET_CHECKPOINT="reset_model"
#load_checkpoint(RESET_CHECKPOINT, model, device)
optim = load_optimizer("SGD", model.parameters(), lr=1e-2)
loss = load_loss_fun("CrossEntropy")
train_set, val_set, test_set = load_data("Waterbirds", reload = False)
data = [train_set, val_set, test_set]
params = {"batch_size":32}
m_params = [params]*3
train_loader, val_loader, test_loader = create_dataloaders(data, m_params)

log, dyn = train_model(
  model=model, 
  train_loader=train_loader, 
  optimizer=optim, 
  loss_fun=loss, 
  n_epochs=20, 
  eval_loader=val_loader, 
  device=device
)
loss, acc = eval_model(model, test_loader, loss,  device)
print("="*20,f"Test set Loss:{loss:.2f} | Acc:{acc:.2f}.","="*20)
all_attr, all_imgs = explain_dataset(train_loader, model, device)
exp_penalty, class_penalty = evaluate_explainations(all_attr, train_set.masks, train_set.y)
print("="*20,f"Exp penalty:{exp_penalty:.2f}.","="*20)
print(class_penalty)

Epoch 4/20:  15%|█▌        | 3/20 [13:33<1:16:47, 271.05s/it, loss=0.5784, acc=0.8419, val_loss=2.3931, val_acc=0.7781]


KeyboardInterrupt: 

In [ ]:
visualize_k_expl(all_attr, all_imgs, train_set, 5)